# Домашнее задание: Feature Engineering на данных Krisha.kz

Цель: Подготовить датасет объявлений о продаже квартир для машинного обучения. Вам нужно создать новые признаки (фичи), которые помогут модели лучше предсказывать цену.


In [2]:
import pandas as pd
import numpy as np

# 1. ЗАГРУЗКА ДАННЫХ
# Не забудь загрузить файл krisha_df.csv в файлы Colab (слева)
try:
    df = pd.read_csv('krisha_df.csv')
    print(f"Данные загружены! Всего объявлений: {df.shape[0]}")
    print("Колонки:", df.columns.tolist())
    display(df.head(3))
except FileNotFoundError:
    print("Файл не найден! Загрузи krisha_df.csv в Colab.")

Данные загружены! Всего объявлений: 20659
Колонки: ['name', 'information', 'address', 'price', 'owner', 'complex_name', 'house_type', 'in_pledge', 'construction_year', 'ceiling_height', 'furniture_info', 'bathroom_info', 'condition', 'area', 'room_count', 'floor', 'floor_count', 'district', 'coordinates_2gis']


,name,information,address,price,owner,complex_name,house_type,in_pledge,construction_year,ceiling_height,furniture_info,bathroom_info,condition,area,room_count,floor,floor_count,district,coordinates_2gis
0,"3-комнатная квартира, 77 м², 1/4 этаж","2018 г.п., санузел раздельный, ✅Полноценная 3 ...",Е 496 10,33000000,Риелтор,kemel,NaN,False,2018,NaN,NaN,раздельный,NaN,77.0,3,1.0,4.0,Есильский р-н,"(51.073887, 71.427446)"
1,"1-комнатная квартира, 40 м², 3/14 этаж","жил. комплекс Jetisu.Lepsi, монолитный дом, 20...",Улы Дала — Ұлы дала,21000000,Риелтор,jetisu.lepsi,монолитный дом,False,2023,3.0,частично мебели,совмещенный,хорошее,40.0,1,3.0,14.0,Есильский р-н,NaN
2,"1-комнатная квартира, 43.8 м², 6/18 этаж","жил. комплекс BURABAY, монолитный дом, 2022 г....",Ж. Нажимеденова 62 — А62,18500000,Риелтор,burabay,монолитный дом,False,2022,2.7,NaN,совмещенный,NaN,43.8,1,6.0,18.0,Алматы р-н,NaN


# ЗАДАЧА 1: Очистка данных (Cleaning)

В колонках floor (этаж), floor_count (этажность) и district (район) есть пропуски. Для нашей задачи эти данные критичны. Если мы не знаем этаж или район, мы не можем правильно оценить квартиру.

Задание: Удалите строки, где есть пустые значения (NaN) в этих трех колонках.

In [3]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Удаляет строки с пропусками (NaN) в колонках 'floor', 'floor_count', 'district'.

    Args:
        df (pd.DataFrame): Исходный датафрейм
    Returns:
        pd.DataFrame: Очищенный датафрейм
    """
    # TODO: Напиши код здесь
    # Подсказка: используй .dropna(subset=[...])
    return df.dropna(subset=['floor', 'floor_count', 'district'])

# --- ТЕСТЫ ---
df_clean = clean_data(df.copy())

assert df_clean['floor'].isna().sum() == 0, "Ошибка: остались пропуски в floor"
assert df_clean['floor_count'].isna().sum() == 0, "Ошибка: остались пропуски в floor_count"
assert df_clean['district'].isna().sum() == 0, "Ошибка: остались пропуски в district"
assert len(df_clean) < len(df), "Ошибка: количество строк не изменилось (ничего не удалено)"

print("Задача 1 пройдена! Мусор удален.")

Задача 1 пройдена! Мусор удален.


# ЗАДАЧА 2: Математические признаки

Общая цена квартиры (price) сильно зависит от площади. Чтобы сравнивать квартиры честно, нужна цена за квадрат.

Задание: Создайте новую колонку price_per_m2, поделив цену (price) на площадь (area).


In [4]:
def add_price_per_m2(df: pd.DataFrame) -> pd.DataFrame:
    """
    Добавляет колонку 'price_per_m2' (Цена / Площадь).

    Args:
        df (pd.DataFrame): Датафрейм
    Returns:
        pd.DataFrame: Датафрейм с новой колонкой
    """
    # TODO: Напиши код здесь
    df['price_per_m2'] = df['price'] / df['area']
    return df

# --- ТЕСТЫ ---
df_clean = add_price_per_m2(df_clean)

assert 'price_per_m2' in df_clean.columns, "Ошибка: колонка price_per_m2 не найдена"
# Проверяем расчет на первой строке
expected_val = df_clean.iloc[0]['price'] / df_clean.iloc[0]['area']
assert np.isclose(df_clean.iloc[0]['price_per_m2'], expected_val), "Ошибка: расчет неверен"

print("Задача 2 пройдена! Цена за квадрат посчитана.")

Задача 2 пройдена! Цена за квадрат посчитана.


#ЗАДАЧА 3: Логические категории (Этажность)

Риелторы знают: первый и последний этажи часто стоят дешевле. Давайте объясним это модели.

Задание: Создайте колонку floor_status с тремя значениями:

- 'First' - если этаж (floor) равен 1.
- 'Last' - если этаж равен этажности дома (floor_count).
- 'Middle' - во всех остальных случаях.


In [8]:
def get_floor_status(row):
    """Вспомогательная функция для определения статуса этажа"""
    # TODO: Напиши логику
    # Если floor == 1 -> вернуть 'First'
    # Если floor == floor_count -> вернуть 'Last'
    # Иначе -> вернуть 'Middle'
    if int(row['floor']) == 1:
        return 'First'
    elif int(row['floor']) == int(row['floor_count']):
        return 'Last'
    else:
        return 'Middle'

def add_floor_category(df: pd.DataFrame) -> pd.DataFrame:
    """
    Добавляет колонку 'floor_status' используя функцию get_floor_status.
    """
    # TODO: Примени функцию через .apply()
    df['floor_status'] = df.apply(get_floor_status, axis=1)
    return df

# --- ТЕСТЫ ---
df_clean = add_floor_category(df_clean)

assert 'floor_status' in df_clean.columns, "Ошибка: колонка floor_status не найдена"
assert df_clean[df_clean['floor'] == 1].iloc[0]['floor_status'] == 'First', "Ошибка: 1 этаж не First"
# Найдем случай, где этаж не первый и не последний (для проверки Middle)
mid_floor = df_clean[(df_clean['floor'] > 1) & (df_clean['floor'] < df_clean['floor_count'])]
if not mid_floor.empty:
    assert mid_floor.iloc[0]['floor_status'] == 'Middle', "Ошибка: средний этаж не Middle"

print("Задача 3 пройдена! Этажи классифицированы.")

Задача 3 пройдена! Этажи классифицированы.


# ЗАДАЧА 4: Работа с датами (Возраст дома)

У нас есть год постройки (construction_year). Модели понятнее, сколько дому лет.

Задание:
- Создайте колонку building_age: отнимите год постройки из текущего года (2025).
- Создайте булеву колонку is_new_building: True, если дому 5 лет или меньше, иначе False.


In [9]:
def add_building_age(df: pd.DataFrame, current_year: int = 2025) -> pd.DataFrame:
    """
    Добавляет:
    1. 'building_age' = current_year - construction_year
    2. 'is_new_building' = True, если возраст <= 5 лет, иначе False
    """
    # TODO: Напиши код здесь
    df['building_age'] = current_year - df['construction_year']
    df['is_new_building'] = df['building_age'] <= 5
    return df

# --- ТЕСТЫ ---
df_clean = add_building_age(df_clean)

assert 'building_age' in df_clean.columns, "Нет колонки building_age"
assert 'is_new_building' in df_clean.columns, "Нет колонки is_new_building"
# Проверка расчета
row = df_clean.iloc[0]
assert row['building_age'] == 2025 - row['construction_year'], "Ошибка в расчете возраста"
assert df_clean[df_clean['building_age'] <= 5].iloc[0]['is_new_building'] == True, "Ошибка в флаге новостройки"

print("Задача 4 пройдена! Возраст и новизна определены.")

Задача 4 пройдена! Возраст и новизна определены.


# ЗАДАЧА 5: One-Hot Encoding (Районы)

Модель не умеет читать слова "Есильский р-н". Ей нужны цифры.

Задание: Примените One-Hot Encoding к колонке district. Используйте pd.get_dummies(). Не забудьте добавить префикс dist.


In [10]:
def encode_districts(df: pd.DataFrame) -> pd.DataFrame:
    """
    Применяет One-Hot Encoding к колонке 'district'.
    Префикс новых колонок должен быть 'dist'.
    """
    # TODO: Используй pd.get_dummies
    return pd.get_dummies(df, columns=['district'], prefix='dist')

# --- ТЕСТЫ ---
df_encoded = encode_districts(df_clean)

# Проверяем, есть ли новые колонки
dist_cols = [c for c in df_encoded.columns if c.startswith('dist_')]
assert len(dist_cols) > 0, "Ошибка: новые колонки dist_... не созданы"
assert df_encoded[dist_cols].dtypes.iloc[0] in [bool, int, 'uint8'], "Ошибка: колонки должны быть числами или булевыми"

print(f"Задача 5 пройдена! Создано {len(dist_cols)} колонок районов.")

Задача 5 пройдена! Создано 5 колонок районов.
